# LangGraph — Explicit Nodes vs LLM-Driven Tool Node

This notebook shows **two approaches side by side**:

**Approach A — Explicit nodes (what you already had):**
```
START → router_node ──(weather)──→ weather_node → END
                    ↘(joke)──────→ joke_node    → END
```

**Approach B — ReAct agent node (new):**
```
START → react_node → END
         (LLM decides whether to call get_weather tool or tell_joke tool)
```

Run both on the same inputs and compare!

In [ ]:
%pip install -q langgraph langchain langchain-openai requests

## 1. Setup

In [15]:
import os
import requests
from typing import TypedDict, Optional, Annotated
import operator

# os.environ["OPENAI_API_KEY"] = "sk-..."       # your OpenAI key
# OPENWEATHER_API_KEY = "your_openweather_key"   # your OpenWeather key

In [16]:
from dotenv import load_dotenv

env_path = "/Users/ahmedibrahim/Desktop/Mids/projects_2026/.env"
load_dotenv(env_path)

True

In [17]:
import os

openai_api_key = os.getenv("OPENAI_API_KEY")
open_weather_api_key = os.getenv("OPENWEATHER_API_KEY")

---
# APPROACH A — Explicit Nodes (your existing graph)

## 2A. State + Nodes

In [18]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

class ExplicitState(TypedDict):
    user_input: str
    intent: str
    city: Optional[str]
    topic: Optional[str]
    final_answer: Optional[str]


def router_node(state: ExplicitState) -> ExplicitState:
#     prompt = f"""Classify this message as 'weather' or 'joke'.
# If weather: INTENT: weather / CITY: <city>
# If joke:    INTENT: joke   / TOPIC: <topic or 'general'>
# Default to joke if unsure.
# Message: {state['user_input']}"""
    
    prompt = f"""Classify this message. Reply on separate lines, no slashes.
If weather:
INTENT: weather
CITY: <city name>

If joke:
INTENT: joke
TOPIC: <topic or 'general'>

Default to joke if unsure.
Message: {state['user_input']}"""

    response = llm.invoke(prompt).content.strip()
    print(f"[Router] →\n{response}\n")

    intent, city, topic = "joke", None, "general"
    for line in response.splitlines():
        line = line.strip()
        if line.startswith("INTENT:"):  intent = line.split(":",1)[1].strip().lower()
        elif line.startswith("CITY:"):  city   = line.split(":",1)[1].strip()
        elif line.startswith("TOPIC:"): topic  = line.split(":",1)[1].strip()

    return {**state, "intent": intent, "city": city, "topic": topic}


def weather_node(state: ExplicitState) -> ExplicitState:
    city = state.get("city", "")
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": open_weather_api_key, "units": "metric"}
    ).json()
    summary = (f"{city}: {resp['weather'][0]['description']}, "
               f"{resp['main']['temp']}°C, humidity {resp['main']['humidity']}%")
    print(f"[Weather Node] {summary}\n")
    answer = llm.invoke(f"User asked: '{state['user_input']}'. Data: {summary}. Short friendly answer.").content
    return {**state, "final_answer": answer}


def joke_node(state: ExplicitState) -> ExplicitState:
    topic = state.get("topic", "general")
    print(f"[Joke Node] topic: {topic}\n")
    answer = llm.invoke(f"Tell one short funny clean joke about: {topic}. Just the joke.").content
    return {**state, "final_answer": answer}

## 3A. Build Explicit Graph

In [19]:
from langgraph.graph import StateGraph, END

def route_decision(state: ExplicitState) -> str:
    return "weather_node" if state["intent"] == "weather" else "joke_node"

builder_a = StateGraph(ExplicitState)
builder_a.add_node("router_node", router_node)
builder_a.add_node("weather_node", weather_node)
builder_a.add_node("joke_node", joke_node)
builder_a.set_entry_point("router_node")
builder_a.add_conditional_edges("router_node", route_decision,
    {"weather_node": "weather_node", "joke_node": "joke_node"})
builder_a.add_edge("weather_node", END)
builder_a.add_edge("joke_node", END)

graph_a = builder_a.compile()
print("Explicit graph compiled ✓")

Explicit graph compiled ✓


---
# APPROACH B — ReAct Agent Node (new)

Here the **LLM decides** which tool to call — no router, no conditional edges.
The same two capabilities (weather + joke) are now wrapped as `@tool`s.

## 2B. Define the Tools

In [20]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Fetch the current weather for a given city.
    Use this when the user asks about weather, temperature, or conditions.
    """
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": open_weather_api_key, "units": "metric"}
    ).json()
    return (f"{city}: {resp['weather'][0]['description']}, "
            f"{resp['main']['temp']}°C, humidity {resp['main']['humidity']}%")


@tool
def tell_joke(topic: str) -> str:
    """
    Generate a short funny joke about a given topic.
    Use this when the user wants a joke, something funny, or wants to laugh.
    """
    return llm.invoke(f"Tell one short funny clean joke about: {topic}. Just the joke.").content


tools = [get_weather, tell_joke]
print("Tools defined:", [t.name for t in tools])

Tools defined: ['get_weather', 'tell_joke']


## 3B. Define the ReAct State + Node

In [21]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage

# ReAct state uses a message list instead of custom fields
# — the LLM reasons through messages to decide what tool to call
class ReactState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]


llm_with_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0.7).bind_tools(tools)
tool_map = {t.name: t for t in tools}


def react_node(state: ReactState) -> ReactState:
    """
    A single ReAct node. The LLM sees the messages and decides:
      - call get_weather tool, OR
      - call tell_joke tool, OR
      - answer directly (no tool needed)
    Then we execute any tool calls and return the final response.
    """
    messages = state["messages"]

    # Step 1: LLM reasons and (maybe) requests a tool call
    ai_response = llm_with_tools.invoke(messages)
    print(f"[ReAct Node] LLM response: {ai_response.content}")
    print(f"[ReAct Node] Tool calls: {ai_response.tool_calls}\n")

    new_messages = [ai_response]

    # Step 2: If the LLM requested tool calls, execute them
    for tool_call in ai_response.tool_calls:
        tool_fn = tool_map[tool_call["name"]]
        tool_result = tool_fn.invoke(tool_call["args"])
        print(f"[ReAct Node] Tool '{tool_call['name']}' result: {tool_result}\n")
        new_messages.append(ToolMessage(
            content=str(tool_result),
            tool_call_id=tool_call["id"]
        ))

    # Step 3: If tools were called, ask the LLM to produce a final answer
    if ai_response.tool_calls:
        final = llm_with_tools.invoke(messages + new_messages)
        new_messages.append(final)

    return {"messages": new_messages}

## 4B. Build ReAct Graph

In [22]:
builder_b = StateGraph(ReactState)
builder_b.add_node("react_node", react_node)
builder_b.set_entry_point("react_node")
builder_b.add_edge("react_node", END)

graph_b = builder_b.compile()
print("ReAct graph compiled ✓")

ReAct graph compiled ✓


---
## 5. Compare Both Approaches Side by Side

In [23]:
def run_both(user_input: str):
    print("=" * 60)
    print(f"INPUT: {user_input}")
    print("=" * 60)

    print("\n--- APPROACH A (Explicit Nodes) ---")
    result_a = graph_a.invoke({"user_input": user_input})
    print("Answer A:", result_a["final_answer"])

    print("\n--- APPROACH B (ReAct Tool Node) ---")
    result_b = graph_b.invoke({"messages": [HumanMessage(content=user_input)]})
    print("Answer B:", result_b["messages"][-1].content)
    print()

In [24]:
run_both("What's the weather in Tokyo?")

INPUT: What's the weather in Tokyo?

--- APPROACH A (Explicit Nodes) ---
[Router] →
INTENT: weather  
CITY: Tokyo

[Weather Node] Tokyo: clear sky, 20.21°C, humidity 64%

Answer A: The weather in Tokyo is nice and clear with a temperature of 20.21°C and 64% humidity. Enjoy your day!

--- APPROACH B (ReAct Tool Node) ---
[ReAct Node] LLM response: 
[ReAct Node] Tool calls: [{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_L0WtS03kNjRejj1lYvU1O0yy', 'type': 'tool_call'}]

[ReAct Node] Tool 'get_weather' result: Tokyo: clear sky, 20.21°C, humidity 64%

Answer B: The weather in Tokyo is clear with a temperature of 20.21°C and humidity at 64%.



In [25]:
run_both("Tell me a joke about Python programmers")

INPUT: Tell me a joke about Python programmers

--- APPROACH A (Explicit Nodes) ---
[Router] →
INTENT: joke  
TOPIC: Python programmers

[Joke Node] topic: Python programmers

Answer A: Why do Python programmers prefer dark mode? Because light attracts bugs!

--- APPROACH B (ReAct Tool Node) ---
[ReAct Node] LLM response: 
[ReAct Node] Tool calls: [{'name': 'tell_joke', 'args': {'topic': 'Python programmers'}, 'id': 'call_MyNMOlhI5OkQaFOAyaEc9wQ3', 'type': 'tool_call'}]

[ReAct Node] Tool 'tell_joke' result: Why do Python programmers prefer dark mode? Because light attracts bugs!

Answer B: Here's a joke for you: 

Why do Python programmers prefer dark mode? Because light attracts bugs!



In [26]:
# Try your own!
run_both("what is the temperature like in Charlotte?")  # change me!

INPUT: what is the temperature like in Charlotte?

--- APPROACH A (Explicit Nodes) ---
[Router] →
INTENT: weather  
CITY: Charlotte

[Weather Node] Charlotte: few clouds, -1.6°C, humidity 55%

Answer A: The temperature in Charlotte is currently -1.6°C with a few clouds and 55% humidity. Stay warm!

--- APPROACH B (ReAct Tool Node) ---
[ReAct Node] LLM response: 
[ReAct Node] Tool calls: [{'name': 'get_weather', 'args': {'city': 'Charlotte'}, 'id': 'call_ByXZQNY9dQfyTnQGDrUNuttA', 'type': 'tool_call'}]

[ReAct Node] Tool 'get_weather' result: Charlotte: few clouds, -1.6°C, humidity 55%

Answer B: The temperature in Charlotte is currently -1.6°C with a few clouds and humidity at 55%.



In [27]:
run_both("Climate of Dhaka")

INPUT: Climate of Dhaka

--- APPROACH A (Explicit Nodes) ---
[Router] →
INTENT: joke  
TOPIC: general

[Joke Node] topic: general

Answer A: Why did the general bring a ladder to the bar? Because he heard the drinks were on the house!

--- APPROACH B (ReAct Tool Node) ---
[ReAct Node] LLM response: Dhaka, the capital of Bangladesh, has a tropical monsoon climate characterized by high temperatures, high humidity, and significant rainfall. Here's a brief overview of its climate:

1. **Seasons**:
   - **Summer (March to June)**: Hot and humid with temperatures often exceeding 30°C (86°F). This season can be uncomfortable due to high humidity.
   - **Monsoon (June to September)**: Characterized by heavy rainfall and thunderstorms. July is usually the wettest month. The temperature remains high, but the rain can provide some relief from the heat.
   - **Post-monsoon (October to November)**: A transitional period with decreasing rainfall and temperatures. The weather is generally pleasant.
 